In [1]:
import subprocess, sys

# ── nvidia-smi ────────────────────────────────────────────────────────────────
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total,compute_cap",
     "--format=csv,noheader"],
    capture_output=True, text=True
)
print("=== GPU ===")
print(result.stdout.strip() if result.returncode == 0 else "nvidia-smi not found")

# ── PyTorch ───────────────────────────────────────────────────────────────────
print("\n=== PyTorch ===")
try:
    import torch
    print(f"Version: {torch.__version__}")
    print(f"CUDA:    {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU:     {torch.cuda.get_device_name(0)}")
        x = torch.rand(1000, 1000, device="cuda")
        torch.matmul(x, x)
        print("Test:    PASSED ✓")
except ImportError:
    print("Not installed")

# ── TensorFlow ────────────────────────────────────────────────────────────────
print("\n=== TensorFlow ===")
try:
    import tensorflow as tf
    print(f"Version: {tf.__version__}")
    gpus = tf.config.list_physical_devices("GPU")
    print(f"GPUs:    {len(gpus)}")
    if gpus:
        with tf.device("/GPU:0"):
            x = tf.random.uniform((1000, 1000))
            tf.matmul(x, x)
        print("Test:    PASSED ✓")
except ImportError:
    print("Not installed")

=== GPU ===
NVIDIA GeForce RTX 5070 Ti, 581.95, 16303 MiB, 12.0

=== PyTorch ===
Version: 2.12.0.dev20260219+cu128
CUDA:    True
GPU:     NVIDIA GeForce RTX 5070 Ti
Test:    PASSED ✓

=== TensorFlow ===


2026-03-23 11:27:44.884689: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-23 11:27:45.186673: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/harris/miniconda3/envs/ai/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to con

Version: 2.20.0
GPUs:    1


W0000 00:00:1774283268.235685     598 gpu_device.cc:2431] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1774283268.238468     598 gpu_device.cc:2431] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1774283268.244415     598 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13089 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5070 Ti, pci bus id: 0000:01:00.0, compute capability: 12.0


Test:    PASSED ✓


In [2]:
import torch
import time

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda")

# large matrices to stress GPU
size = 10000

print("\nAllocating tensors on GPU...")
a = torch.rand(size, size, device=device)
b = torch.rand(size, size, device=device)

# warmup (important for GPU timing)
torch.matmul(a, b)
torch.cuda.synchronize()

print("Running benchmark...")

start = time.time()

c = torch.matmul(a, b)

torch.cuda.synchronize()
end = time.time()

print(f"Matrix size: {size} x {size}")
print(f"Execution time: {end - start:.4f} seconds")

print("\nGPU memory usage:")
print("Allocated:", torch.cuda.memory_allocated() / 1024**3, "GB")
print("Reserved :", torch.cuda.memory_reserved() / 1024**3, "GB")

PyTorch version: 2.12.0.dev20260219+cu128
CUDA available: True

Allocating tensors on GPU...
Running benchmark...
Matrix size: 10000 x 10000
Execution time: 0.0607 seconds

GPU memory usage:
Allocated: 1.1270751953125 GB
Reserved : 1.138671875 GB
